In [ ]:
import torch
from tokenizers import Tokenizer
from tokenizers.models import BPE

from tokenizers.pre_tokenizers import Whitespace
from tokenizers.trainers import BpeTrainer
from torch.utils.data import Dataset, DataLoader, random_split
from torch import nn

import math
import pandas as pd

## Read the data

In [ ]:
url = "https://github.com/PhilChodrow/PIC16B/blob/master/datasets/star_trek_scripts.json?raw=true"
star_trek_scripts = pd.read_json(url)

star_trek_scripts = star_trek_scripts["DS9"].str.replace("\n\n\n\n\n\nThe Deep Space Nine Transcripts -", "")
star_trek_scripts = star_trek_scripts.str.split("\n\n\n\n\n\n\n").str.get(-2)
star_trek_scripts = star_trek_scripts[pd.isna(star_trek_scripts) == False].tolist()

text = "\n\n".join(star_trek_scripts)

for char in ['\xa0', 'à', 'é', "}", "{"]:
    text = text.replace(char, "")


text = text[:128000]  # use a subset for speed

In [ ]:
class CharTokenizer():
    def __init__(self, text):
        self.chars = sorted(set(text))
        self.char_to_id = {ch: i for i, ch in enumerate(self.chars)}
        self.id_to_char = {i: ch for ch, i in self.char_to_id.items()}

    def encode(self, s):
        return [self.char_to_id[c] for c in s]

    def decode(self, ids):
        return ''.join([self.id_to_char[i] for i in ids])

    def get_vocab_size(self):
        return len(self.chars)

    def get_vocab(self):
        return self.char_to_id


tokenizer = CharTokenizer(text)
ids = tokenizer.encode(text)

## Data set, data loader

In [ ]:
class SequenceDataset(Dataset):
    def __init__(self, token_ids, seq_len):
        self.seq_len = seq_len
        self.data = torch.tensor(token_ids, dtype=torch.long)
        self.n = len(self.data) - seq_len

    def __len__(self):
        return max(self.n, 0)

    def __getitem__(self, idx):
        x = self.data[idx:idx + self.seq_len]
        y = self.data[idx + 1 + self.seq_len]
        return x, y


def data_loaders_from_dataset(dataset, batch_size=32, val_split=0.05):
    train_size = int((1 - val_split) * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = random_split(dataset, [train_size, val_size])
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, drop_last=True)
    val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False, drop_last=True)
    return train_loader, val_loader

data = SequenceDataset(ids, seq_len=128)


train_loader, val_loader = data_loaders_from_dataset(SequenceDataset(ids, seq_len=128))

## Minimal Model

In [ ]:
# positional encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=10000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe)

    def forward(self, x):
        return x + self.pe[:x.size(1)].unsqueeze(0)

class MiniTransformerLM(nn.Module):
    def __init__(self, vocab_size, d_model=256, nhead=4, num_layers=2, dim_ff=512, dropout=0.1):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = PositionalEncoding(d_model)
        encoder_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_ff, dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers)
        self.lm_head = nn.Linear(d_model, vocab_size)
    

    def forward(self, x):
        mask = torch.triu(torch.ones(x.size(1), x.size(1), device=x.device), diagonal=1).bool()
        h = self.embed(x)
        # print(h.shape)

        h = self.pos(h)
        # print(h.shape)
        h = self.encoder(h, mask=mask)

        # print(h.shape)

        

        h = self.lm_head(h)

        h = h.sum(dim=1)
        return h

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
vocab_size = tokenizer.get_vocab_size()
model = MiniTransformerLM(vocab_size).to(device)

In [ ]:
X, y = next(iter(train_loader))
X, y = X.to(device), y.to(device)
logits = model(X)


In [ ]:
def generate_text(model, tokenizer, prompt, max_length=50, temperature=1.0):
    model.eval()
    tokens = tokenizer.encode(prompt)
    input_ids = torch.tensor(tokens, dtype=torch.long).unsqueeze(0).to(device)

    generated_ids = input_ids.tolist()[0]

    for _ in range(max_length):
        with torch.no_grad():
            logits = model(input_ids)
            logits = logits / temperature
            
            probs = torch.softmax(logits, -1)
            next_token_id = torch.multinomial(probs, num_samples=1).item()

            generated_ids.append(next_token_id)
            input_ids = torch.tensor(generated_ids[-128:], dtype=torch.long).unsqueeze(0).to(device)

    generated_text = tokenizer.decode(generated_ids)
    generated_text = generated_text.replace("NEWLINE", "\n")
    return generated_text

prompt = "SISKO:"
generated_text = generate_text(model, tokenizer, prompt, max_length=100, temperature = 1.0)


In [ ]:
num_epochs = 10
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.CrossEntropyLoss()

In [ ]:
generated_text = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0
    for i, (X, y) in enumerate(train_loader):
        X, y = X.to(device), y.to(device)

        optimizer.zero_grad()
        logits = model(X)
        loss = criterion(logits.view(-1, vocab_size), y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        if i % 10 == 0:
            print(f"Batch {i}, Loss: {loss.item():.4f}")
            text = generate_text(model, tokenizer, prompt, max_length=100, temperature=5.0)
            print(text)
            generated_text.append(text)

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch + 1}/{num_epochs}, Loss: {avg_loss:.4f}")

In [ ]:
generate_text(model, tokenizer, prompt, max_length=100, temperature=2)